In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV
from statsmodels.tsa.stattools import adfuller

In [10]:
df = pd.read_csv('grocery_dataset_eda.csv')
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')
df = df[(df['date'] >= '2005-01-01') & (df['date'] <= '2025-01-01')]
df = df.reset_index().set_index('date').drop(columns = 'index')
df

,gdp,cons_exp,bus_inv,gov_exp,net_exports,imports,exports,rdi,cpi,unrate,...,wages_retail_yoy_lag3,wages_retail_yoy_lag4,wages_retail_yoy_lag5,wages_retail_yoy_lag6,wages_retail_yoy_lag7,wages_retail_yoy_lag8,wages_retail_yoy_lag9,wages_retail_yoy_lag10,wages_retail_yoy_lag11,wages_retail_yoy_lag12
date,,,,,,,,,,,,,,,,,,,,,
2005-01-31,15844.727,NaN,440.005,4201.029,-701.438,2177.25,1430.827,11226.5,191.600,5.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-02-28,15844.727,NaN,440.005,4201.029,-701.438,2177.25,1430.827,11229.0,192.400,5.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-03-31,15844.727,NaN,440.005,4201.029,-701.438,2177.25,1430.827,11268.8,193.100,5.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-04-30,15922.782,NaN,381.969,4240.850,-697.998,2177.25,1462.813,11304.2,193.700,5.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-05-31,15922.782,NaN,381.969,4240.850,-697.998,2177.25,1462.813,11352.7,193.600,5.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-31,23400.294,16088.6,938.835,10387.702,-1069.229,NaN,2638.199,17494.6,314.131,4.2,...,2.732240,2.099076,2.743774,2.524190,2.931181,3.371746,2.425532,3.178694,4.197317,3.987863
2024-09-30,23400.294,16173.4,938.835,10387.702,-1069.229,NaN,2638.199,17519.6,314.851,4.1,...,2.260360,2.732240,2.099076,2.743774,2.524190,2.931181,3.371746,2.425532,3.178694,4.197317
2024-10-31,23542.349,16194.7,808.793,10475.498,-1052.658,NaN,2637.171,17568.5,315.564,4.1,...,2.130326,2.260360,2.732240,2.099076,2.743774,2.524190,2.931181,3.371746,2.425532,3.178694


In [11]:
# Count total missing values per column
missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0]
missing_counts

gdp                        2
cons_exp                  24
bus_inv                    2
gov_exp                    2
net_exports                2
                          ..
wages_retail_yoy_lag8     34
wages_retail_yoy_lag9     35
wages_retail_yoy_lag10    36
wages_retail_yoy_lag11    37
wages_retail_yoy_lag12    38
Length: 265, dtype: int64

In [12]:
# Count infs
inf_counts = df.isin([np.inf, -np.inf]).sum()
inf_counts = inf_counts[inf_counts>0]
inf_counts

Series([], dtype: int64)

In [13]:
for col in df.columns:
    print(col)

gdp
cons_exp
bus_inv
gov_exp
net_exports
imports
exports
rdi
cpi
unrate
initial_claims
continued_clains
oil_prices
cpi_fah
grocery_sales
restaurant_sales
ppi_final_food
ppi_farm_products
ppi_food_feed
ppi_fin_cons_food
ppi_food_mfg
ppi_grocery
wages_retail
home_price
cons_sent
savings
cons_debt_di
credit_card_delinq
mortgage_delinq
us_grocery_units
rdi_adj
grocery_sales_lag1
covid1
covid2
gdp_diff
cons_exp_diff
bus_inv_diff
gov_exp_diff
net_exports_diff
imports_diff
exports_diff
rdi_diff
cpi_diff
unrate_diff
initial_claims_diff
continued_clains_diff
oil_prices_diff
cpi_fah_diff
grocery_sales_diff
restaurant_sales_diff
ppi_final_food_diff
ppi_farm_products_diff
ppi_food_feed_diff
ppi_fin_cons_food_diff
ppi_food_mfg_diff
ppi_grocery_diff
wages_retail_diff
home_price_diff
cons_sent_diff
savings_diff
cons_debt_di_diff
credit_card_delinq_diff
mortgage_delinq_diff
us_grocery_units_diff
rdi_adj_diff
grocery_sales_lag1_diff
gdp_yoy
cons_exp_yoy
bus_inv_yoy
gov_exp_yoy
net_exports_yoy
imports_y

In [14]:
df.to_csv('temp.csv')

### RIDGE ###


In [7]:
# Define Features (X) and Target (y)
X = df[['grocery_sales_lag1', 'cpi_fah', 'home_price', 'rdi_adj', 'credit_card_delinq', 'covid1', 'covid2']]
y = df['grocery_sales'] 

# Handle missing values (fill with median)
X = X.fillna(X.median())

# Standardize features (important for Ridge Regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Standardize the features

# Convert back to a DataFrame to retain column names
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)

# Define training data (2004-2023) and test data (2024)
X_train = X_scaled_df.loc[:'2023-12-31']  # Features for 2004-2023
y_train = y.loc[:'2023-12-31']            # Target for 2004-2023

X_test = X_scaled_df.loc['2024-01-01':]   # Features for 2024
y_test = y.loc['2024-01-01':]             # Target for 2024

# Print train/test sizes to verify
print(f"Train Size: {X_train.shape}, Test Size: {X_test.shape}")


Train Size: (228, 7), Test Size: (12, 7)


C:\Users\erick\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
C:\Users\erick\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
C:\Users\erick\anaconda3\Lib\site-packages\sklearn\utils\extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [8]:
# Define a range of alpha values to test
alpha_values = np.logspace(-3, 3, 100)  # Values from 0.001 to 1000

# Ridge Regression with cross-validation
ridge_cv = RidgeCV(alphas=alpha_values, store_cv_results=True)
ridge_cv.fit(X_train, y_train)

# Best alpha value selected
best_alpha = ridge_cv.alpha_
print(f"🔍 Best Alpha Found: {best_alpha}")

# Train Ridge model with best alpha
ridge = Ridge(alpha=3.7649)  # Directly using best value
ridge.fit(X_train, y_train)

ridge_train_forecast = pd.Series(ridge.predict(X_train), index=y_train.index)
ridge_test_forecast = pd.Series(ridge.predict(X_test), index=y_test.index)



ValueError: Input X contains NaN.
_RidgeGCV does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
ridge_coefs = pd.DataFrame({
    "Feature": ['US Grocery Sales_lag1', 'CPI (Food at Home)', 'Avg Home Price', 'Real Disposable Income adj', 'Credit Card Delinquency', 'covid1', 'covid2'],
    "Coefficient": ridge.coef_
}).sort_values(by="Coefficient", ascending=False)

print("📌 Ridge Regression Coefficients:")
print(ridge_coefs)


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

def create_forecast_results_df(y_train, y_test, train_forecast, test_forecast):
    """
    Creates a standardized forecast DataFrame and computes performance metrics.

    Parameters:
        y_train (pd.Series): Actual sales in the training period
        y_test (pd.Series): Actual sales in the test period
        train_forecast (pd.Series): Model's forecast for the training period
        test_forecast (pd.Series): Model's forecast for the test period

    Returns:
        forecast_df (pd.DataFrame): DataFrame containing actuals, forecasts, and residuals
        metrics (dict): Dictionary containing R², MSE, MAE, and MAPE
    """

    # Compute residuals
    train_residuals = y_train - train_forecast
    test_residuals = y_test - test_forecast

    # Force missing values to NaN instead of None
    forecast_df = pd.DataFrame({
        "Actual_Sales": pd.concat([y_train, y_test]),
        "Train_Forecast": pd.concat([train_forecast.loc[y_train.index], pd.Series([np.nan] * len(y_test), index=y_test.index)]),
        "Test_Forecast": pd.concat([pd.Series([np.nan] * len(y_train), index=y_train.index), test_forecast]),
        "Train_Residuals": pd.concat([train_residuals, pd.Series([np.nan] * len(y_test), index=y_test.index)]),
        "Test_Residuals": pd.concat([pd.Series([np.nan] * len(y_train), index=y_train.index), test_residuals])
    }).astype(float)  # 🔹 Ensure all columns remain float

    # Compute Performance Metrics
    metrics = {
        "Train R²": r2_score(y_train, train_forecast),
        "Test R²": r2_score(y_test, test_forecast),
        "Train MSE": mean_squared_error(y_train, train_forecast),
        "Test MSE": mean_squared_error(y_test, test_forecast),
        "Train MAE": mean_absolute_error(y_train, train_forecast),
        "Test MAE": mean_absolute_error(y_test, test_forecast),
        "Train MAPE": np.mean(np.abs((y_train - train_forecast) / y_train)) * 100,
        "Test MAPE": np.mean(np.abs((y_test - test_forecast) / y_test)) * 100
    }

    return forecast_df, metrics



In [ ]:
# Generate Ridge forecast DataFrame with metrics
ridge_forecast_df, ridge_metrics = create_forecast_results_df(
    y_train = y_train,
    y_test = y_test,
    train_forecast = ridge_train_forecast,
    test_forecast = ridge_test_forecast
)

# Print first few rows of the forecast results
print(ridge_forecast_df.tail(12))

# Print Ridge Regression performance metrics
print("📊 Ridge Regression Performance Metrics:")
for key, value in ridge_metrics.items():
    print(f"{key}: {value:.4f}")


In [ ]:
# Plot First-Difference Forecast vs Actual with Confidence Interval Lines
fig, ax = plt.subplots(figsize=(12, 6))

# Plot actual, train, and test forecast
ax.plot(ridge_forecast_df.index, ridge_forecast_df['Actual_Sales'], label="Actual Sales", color="blue")
ax.plot(ridge_forecast_df.index, ridge_forecast_df['Train_Forecast'], label="Train Forecast", color="red", linestyle="dashed")
ax.plot(ridge_forecast_df.index, ridge_forecast_df['Test_Forecast'], label="Test Forecast", color="green", linestyle="dashed")

# Formatting
ax.set_xlabel("Month")
ax.set_ylabel("Grocery Sales")
ax.set_title("Ridge: Forecast vs Actual")
ax.legend()

plt.show()

In [ ]:
df.index

### OLS ###

In [ ]:
from sklearn.linear_model import LinearRegression

# Define training data (2004-2023) and test data (2024)
# X_train = X_scaled_df.loc[:'2023-12-31']  # Features for 2004-2023
# y_train = y.loc[:'2023-12-31']            # Target for 2004-2023

# X_test = X_scaled_df.loc['2024-01-31':]   # Features for 2024
# y_test = y.loc['2024-01-31':]             # Target for 2024

# Train standard linear regression model
lin_reg = LinearRegression().fit(X_train, y_train)

# Predict for training period (2004-2023) and convert to Pandas Series
ols_train_forecast = pd.Series(lin_reg.predict(X_train), index=y_train.index, name="OLS_Train_Forecast")

# Predict for test period (2024) and convert to Pandas Series
ols_test_forecast = pd.Series(lin_reg.predict(X_test), index=y_test.index, name="OLS_Test_Forecast")

In [ ]:
# Generate Ridge forecast DataFrame with metrics
ols_forecast_df, ols_metrics = create_forecast_results_df(
    y_train = y_train,
    y_test = y_test,
    train_forecast=ols_train_forecast,
    test_forecast=ols_test_forecast
)

# Print first few rows of the forecast results
print(ols_forecast_df.tail(12))

# Print Ridge Regression performance metrics
print("📊 OLS Regression Performance Metrics:")
for key, value in ols_metrics.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# Plot 
fig, ax = plt.subplots(figsize=(12, 6))

# Plot actual, train, and test forecast
ax.plot(ols_forecast_df.index, ols_forecast_df['Actual_Sales'], label="Actual Sales", color="blue")
ax.plot(ols_forecast_df.index, ols_forecast_df['Train_Forecast'], label="Train Forecast", color="red", linestyle="dashed")
ax.plot(ols_forecast_df.index, ols_forecast_df['Test_Forecast'], label="Test Forecast", color="green", linestyle="dashed")

# Formatting
ax.set_xlabel("Month")
ax.set_ylabel("Grocery Sales")
ax.set_title("OLS: Forecast vs Actual")
ax.legend()

plt.show()

### GRADIENT BOOST ###

In [ ]:
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

target_col = "US Grocery Sales"
from sklearn.linear_model import LinearRegression

# # Define training data (2004-2023) and test data (2024)
# X_train = X_scaled_df.loc[:'2023-12-31']  # Features for 2004-2023
# y_train = y.loc[:'2023-12-31']            # Target for 2004-2023

# X_test = X_scaled_df.loc['2024-01-31':]   # Features for 2024
# y_test = y.loc['2024-01-31':]             # Target for 2024

# Train XG Boost
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.5,
    max_depth=3,
    reg_alpha = 1,
    reg_lambda = 1,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Predict for training period (2004-2023)
y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# Convert NumPy arrays to pandas Series with correct index
y_train_pred_xgb = pd.Series(y_train_pred_xgb, index=y_train.index)
y_test_pred_xgb = pd.Series(y_test_pred_xgb, index=y_test.index)

xgb_forecast_df, xgb_metrics = create_forecast_results_df(
    y_train = y_train,
    y_test = y_test,
    train_forecast = y_train_pred_xgb,
    test_forecast = y_test_pred_xgb
)

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

xgb_metrics = {
    "Train R²": r2_score(y_train, y_train_pred_xgb),
    "Test R²": r2_score(y_test, y_test_pred_xgb),
    "Train MSE": mean_squared_error(y_train, y_train_pred_xgb),
    "Test MSE": mean_squared_error(y_test, y_test_pred_xgb),
    "Train MAE": mean_absolute_error(y_train, y_train_pred_xgb),
    "Test MAE": mean_absolute_error(y_test, y_test_pred_xgb),
    "Train MAPE": mape(y_train, y_train_pred_xgb),
    "Test MAPE": mape(y_test, y_test_pred_xgb),
}

xgb_forecast_df

In [ ]:
# Plot First-Difference Forecast vs Actual with Confidence Interval Lines
fig, ax = plt.subplots(figsize=(12, 6))

# Plot actual, train, and test forecast
ax.plot(xgb_forecast_df.index, xgb_forecast_df['Actual_Sales'], label="Actual Sales", color="blue")
ax.plot(xgb_forecast_df.index, xgb_forecast_df['Train_Forecast'], label="Train Forecast", color="red", linestyle="dashed")
ax.plot(xgb_forecast_df.index, xgb_forecast_df['Test_Forecast'], label="Test Forecast", color="green", linestyle="dashed")

# Formatting
ax.set_xlabel("Month")
ax.set_ylabel("Grocery Sales")
ax.set_title("Forecast vs Actual")
ax.legend()

plt.show()

In [ ]:
# Create a dictionary of model metrics
metrics_data = {
    "Model": ["Ridge", "OLS", "XGBOOST"],
    "Train R²": [ridge_metrics["Train R²"], ols_metrics["Train R²"], xgb_metrics["Train R²"]],
    "Test R²": [ridge_metrics["Test R²"], ols_metrics["Test R²"], xgb_metrics["Test R²"]],
    "Train MSE": [ridge_metrics["Train MSE"], ols_metrics["Train MSE"], xgb_metrics["Train MSE"]],
    "Test MSE": [ridge_metrics["Test MSE"], ols_metrics["Test MSE"], xgb_metrics["Test MSE"]],
    "Train MAE": [ridge_metrics["Train MAE"], ols_metrics["Train MAE"], xgb_metrics["Train MAE"]],
    "Test MAE": [ridge_metrics["Test MAE"], ols_metrics["Test MAE"], xgb_metrics["Test MAE"]],
    "Train MAPE": [ridge_metrics["Train MAPE"], ols_metrics["Train MAPE"], xgb_metrics["Train MAPE"]],
    "Test MAPE": [ridge_metrics["Test MAPE"], ols_metrics["Test MAPE"], xgb_metrics["Test MAPE"]]
}

# Convert to DataFrame
metrics_df = pd.DataFrame(metrics_data)

metrics_df

### CPI FAH FORECAST ###

In [ ]:
df_cpi = df[df.index >= '2012-01-01']

In [ ]:
# Define dependent variable (Y)
y = df_cpi["CPI (Food at Home)_yoy"]

# Define independent variables (X)
X = df_cpi[["PPI Food and Feed_yoy_lag4", "PPI Final Food_yoy_lag5", "Oil Prices_yoy_lag9", "PPI Grocery_yoy"]]

# Count total missing values per column
missing_counts_y = y.isna().sum()
missing_counts_y = missing_counts_y[missing_counts_y > 0]

# Count total missing values per column
missing_counts_X = X.isna().sum()
missing_counts_X = missing_counts_X[missing_counts_X > 0]

print(missing_counts_y, missing_counts_X) 


In [ ]:
# Plot First-Difference Forecast vs Actual with Confidence Interval Lines
fig, ax = plt.subplots(figsize=(12, 6))

# Plot actual, train, and test forecast
ax.plot(df_cpi.index, df_cpi['CPI (Food at Home)_yoy'], label="CPI FAH YOY", color="blue")
ax.plot(df_cpi.index, df_cpi['PPI Food and Feed_yoy_lag4'], label="PPI Food and Feed lag4", color="red", linestyle="dashed")
ax.plot(df_cpi.index, df_cpi['PPI Grocery_yoy'], label="PPI Grocery", color="green", linestyle="dashed")
ax.plot(df_cpi.index, df_cpi['PPI Final Food_yoy_lag5'], label="PPI Final Food_yoy_lag5 ", color="black", linestyle="dashed")
# Formatting
ax.set_xlabel("Month")
ax.set_ylabel("CPI FAH YOY")
ax.set_title("CPI FAH YOY vs PPI FOOD and FEED Lag4")
ax.legend()

plt.show()

In [ ]:
# Add a constant term for the intercept
X = sm.add_constant(X)

# Fit the model
cpi_model = sm.OLS(y, X).fit()

# Display the summary
print(cpi_model.summary())

In [ ]:
from sklearn.linear_model import LinearRegression

# Define dependent variable (Y)
y = df_cpi["CPI (Food at Home)_yoy"]

# Define independent variables (X)
X = df_cpi[["PPI Food and Feed_yoy_lag4", "PPI Final Food_yoy_lag5", "Oil Prices_yoy_lag9", "PPI Grocery_yoy"]]

# Define training data (2004-2023) and test data (2024)
X_train = X.loc[:'2024-08-1']  # Ensures only selected columns are included
y_train = y.loc[:'2024-08-01']  # Target for 2004-2023

X_test = X.loc['2024-09-01':]  # Ensures only selected columns are included
y_test = y.loc['2024-09-01':]  # Target for 2024


In [ ]:
print(X_train.isna().sum())  # Count NaNs in training data
print(X_test.isna().sum())   # Count NaNs in test data


In [ ]:
# Train standard linear regression model
lin_reg = LinearRegression().fit(X_train, y_train)

# Predict for training period (2004-2023) and convert to Pandas Series
cpi_train_fcst = pd.Series(lin_reg.predict(X_train), index=y_train.index, name="CPI_Train_Forecast")

# Predict for test period (2024) and convert to Pandas Series
cpi_test_fcst = pd.Series(lin_reg.predict(X_test), index=y_test.index, name="CPI_Test_Forecast")

In [ ]:
# Generate forecast DataFrame with metrics
cpi_forecast_df, cpi_metrics = create_forecast_results_df(
    y_train = y_train,
    y_test = y_test,
    train_forecast=cpi_train_fcst,
    test_forecast=cpi_test_fcst
)

# Print first few rows of the forecast results
print(cpi_forecast_df.tail(12))

# Print Ridge Regression performance metrics
print("📊 OLS Regression Performance Metrics:")
for key, value in cpi_metrics.items():
    print(f"{key}: {value:.2f}")

In [ ]:
# Plot First-Difference Forecast vs Actual with Confidence Interval Lines
fig, ax = plt.subplots(figsize=(12, 6))

# Plot actual, train, and test forecast
ax.plot(cpi_forecast_df.index, cpi_forecast_df['Actual_Sales'], label="Actual Sales", color="blue")
ax.plot(cpi_forecast_df.index, cpi_forecast_df['Train_Forecast'], label="Train Forecast", color="red", linestyle="dashed")
ax.plot(cpi_forecast_df.index, cpi_forecast_df['Test_Forecast'], label="Test Forecast", color="green", linestyle="dashed")

# Formatting
ax.set_xlabel("Month")
ax.set_ylabel("CPI Food at Home")
ax.set_title("CPI FAH: Forecast vs Actual")
ax.legend()

plt.show()


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

# Make sure the index is datetime and sorted
df_cpi = df_cpi.sort_index()
df_cpi.index = pd.to_datetime(df_cpi.index)  # If it's not already datetime

results = []

# Define unique months in the index
all_months = df_cpi.index.to_period('M').unique()  # PeriodIndex for monthly grouping

# Define rolling parameters
n_periods = 24  # last 24 months
test_window = 5

for i in range(len(all_months) - n_periods, len(all_months) - test_window + 1):
    test_months = all_months[i:i + test_window]
    train_months = all_months[:i]  # everything before the test window
    
    # Filter data by index (monthly PeriodIndex)
    train_df = df_cpi[df_cpi.index.to_period('M').isin(train_months)]
    test_df = df_cpi[df_cpi.index.to_period('M').isin(test_months)]
    
    # Fit your model (replace with your actual model and features)
    model = LinearRegression()
    X_train = train_df[["PPI Food and Feed_yoy_lag4", "PPI Final Food_yoy_lag5", "Oil Prices_yoy_lag9", "PPI Grocery_yoy"]]
    y_train = train_df["CPI (Food at Home)_yoy"]
    X_test = test_df[["PPI Food and Feed_yoy_lag4", "PPI Final Food_yoy_lag5", "Oil Prices_yoy_lag9", "PPI Grocery_yoy"]]
    y_test = test_df["CPI (Food at Home)_yoy"]
    
    model.fit(X_train, y_train)
    forecast = model.predict(X_test)
    
    # Store performance
    mae = mean_absolute_error(y_test, forecast)
    
    results.append({
        'test_start': test_months[0].strftime('%Y-%m'),
        'test_end': test_months[-1].strftime('%Y-%m'),
        'MAE': mae,
        'Actual': y_test.values,
        'Forecast': forecast
    })

# Turn results into a DataFrame
rolling_results_df = pd.DataFrame(results)


In [ ]:
plt.plot(rolling_results_df['test_start'], rolling_results_df['MAE'])
plt.xticks(rotation=45)
plt.title('Rolling 5-Month Forecast Error (MAE)')
plt.xlabel('Test Window Start Month')
plt.ylabel('Mean Absolute Error')
plt.tight_layout()
plt.show()

In [ ]:
#MERGE The three forecast dfs with the main dfs

# Remove Actual Sales from each df or we will get a duplication error
for forecast_df in [ridge_forecast_df, ols_forecast_df, xgb_forecast_df, cpi_forecast_df]:
    forecast_df.drop(columns="Actual_Sales", inplace=True)

#Rename columns
ridge_forecast_df = ridge_forecast_df.rename(columns={
    "Train_Forecast": "Ridge_Train_Forecast",
    "Test_Forecast": "Ridge_Test_Forecast",
    "Train_Residuals": "Ridge_Train_Residuals",
    "Test_Residuals": "Ridge_Test_Residuals"
})

ols_forecast_df = ols_forecast_df.rename(columns={
    "Train_Forecast": "OLS_Train_Forecast",
    "Test_Forecast": "OLS_Test_Forecast",
    "Train_Residuals": "OLS_Train_Residuals",
    "Test_Residuals": "OLS_Test_Residuals"
})

xgb_forecast_df = xgb_forecast_df.rename(columns={
    "Train_Forecast": "XGB_Train_Forecast",
    "Test_Forecast": "XGB_Test_Forecast",
    "Train_Residuals": "XGB_Train_Residuals",
    "Test_Residuals": "XGB_Test_Residuals"
})

cpi_forecast_df = cpi_forecast_df.rename(columns={
        "Train_Forecast": "CPI_Train_Forecast",
        "Test_Forecast": "CPI_Test_Forecast",
        "Train_Residuals": "CPI_Train_Residuals",
        "Test_Residuals": "CPI_Test_Residuals"
})

df = df.merge(ridge_forecast_df, how="left", left_index=True, right_index=True)
df = df.merge(ols_forecast_df, how="left", left_index=True, right_index=True)
df = df.merge(xgb_forecast_df, how="left", left_index=True, right_index=True)
df = df.merge(cpi_forecast_df, how="left", left_index=True, right_index=True)

df

In [ ]:
df.to_csv('forecast_df.csv')
metrics_df.to_csv('model_performance_metrics.csv')